# Python for AI — Class 13
### NumPy — arrays, the language of AI data

Week 3 starts the data libraries. **NumPy** is the foundation under all of them: Pandas, scikit-learn, PyTorch — every one stores its numbers in NumPy-style arrays. An image is an array of pixels, a dataset is an array of rows, a neural network is arrays of weights. Today you learn to create, index and slice arrays, and to do maths on a million numbers **without writing a loop**.

**How to use this notebook:** run each cell with the play button, or `Shift + Enter`.

Cells marked **BREAKS ON PURPOSE** are *supposed* to show a red error. Cells marked **WRONG ON PURPOSE** run fine and give the *wrong answer* — those are the dangerous ones. Don't fix either before class; that is the lesson.

In [7]:
# Colab has NumPy already. On your own machine: pip install numpy
import numpy as np          # 'np' is the universal nickname - everyone uses it
print(np.__version__)

1.26.4


# 1. Why not just use lists?

Say you want to add 10% tax to every price. With a list you need a loop. With an array you write the maths once and NumPy applies it to every element:

In [1]:
prices_list = [1200, 850, 400, 3500]

# WRONG ON PURPOSE - list maths doesn't mean what you think
print(prices_list * 2)            # repeats the list, doesn't double the prices
print(prices_list + [100])        # glues on an element, doesn't add 100

[1200, 850, 400, 3500, 1200, 850, 400, 3500]
[1200, 850, 400, 3500, 100]


In [8]:
prices = np.array([1200, 850, 400, 3500])

print(prices * 2)                 # every element doubled
print(prices + 100)               # every element + 100
print(prices * 1.10)              # 10% tax on everything, no loop

[2400 1700  800 7000]
[1300  950  500 3600]
[1320.  935.  440. 3850.]


To a list, `*` and `+` mean *repeat* and *join*. To an array they mean **maths on every element**. That's called a **vectorized** operation. It's also far faster — let's measure it.

In [ ]:
import time

n = 1_000_000
big_list = list(range(n))
big_array = np.arange(n)

start = time.time()
squared_list = [x * x for x in big_list]
list_time = time.time() - start

start = time.time()
squared_array = big_array * big_array
array_time = time.time() - start

print(f"list  : {list_time:.4f} s")
print(f"array : {array_time:.4f} s")
print(f"NumPy is about {list_time / array_time:.0f}x faster")

The loop runs in Python one number at a time; NumPy runs in compiled C over a block of memory where every element has the same type. Training a model means billions of these operations — that speed-up is the reason AI in Python is possible at all.

# 2. Creating arrays and dimensions

An array has a **shape** — how many elements along each dimension (axis):

```
1-D  [1, 2, 3]                shape (3,)       a vector  - one student's marks
2-D  [[1, 2, 3],              shape (2, 3)     a matrix  - rows = students, cols = subjects
      [4, 5, 6]]
3-D  a stack of matrices      shape (h, w, 3)  a colour image: height x width x RGB
```

In [10]:
v = np.array([80, 72, 91])                     # 1-D
m = np.array([
    [80, 72, 91],
    [65, 88, 70]
])                   # 2-D: 2 students x 3 subjects

# for name, arr in [("v", v), ("m", m)]:
#     print(f"{name}: ndim={arr.ndim}  shape={arr.shape}  size={arr.size}  dtype={arr.dtype}")

In [11]:
# Ready-made constructors - you'll use these more than np.array
print(np.zeros((2, 3)))               # 2x3 of zeros
print(np.ones(4))                     # four ones
print(np.arange(0, 10, 2))            # like range(): start, stop, step
print(np.linspace(0, 1, 5))           # 5 evenly spaced numbers from 0 to 1 (inclusive)
print(np.random.randint(0, 100, size=(2, 4)))   # random whole numbers, 2x4

[[0. 0. 0.]
 [0. 0. 0.]]
[1. 1. 1. 1.]
[0 2 4 6 8]
[0.   0.25 0.5  0.75 1.  ]
[[79  4 51 42]
 [46 86 29  8]]


In [14]:
# reshape: same numbers, new shape
a = np.arange(12)
print(a)
print(a.reshape(3, 4))
print(a.reshape(4, -1))               # -1 = "work this one out for me"

[ 0  1  2  3  4  5  6  7  8  9 10 11]
[[ 0  1  2  3]
 [ 4  5  6  7]
 [ 8  9 10 11]]
[[ 0  1  2]
 [ 3  4  5]
 [ 6  7  8]
 [ 9 10 11]]


In [ ]:
# BREAKS ON PURPOSE - the element count must match
np.arange(12).reshape(5, 3)           # 5 x 3 = 15, but we only have 12

**ValueError: cannot reshape array of size 12 into shape (5,3).** Shape errors are *the* most common error in ML code. Whenever something breaks, the first thing to do is `print(x.shape)`.

In [ ]:
# WRONG ON PURPOSE - an array has ONE type, and it can silently convert
marks = np.array([80, 72, 91])
marks[0] = 85.7                  # we meant to store 85.7...
print(marks)                     # ...but the array is int, so it's truncated to 85

mixed = np.array([1, 2, "3"])
print(mixed, mixed.dtype)        # one string turns everything into strings

Lists hold anything; arrays hold **one** `dtype`. Put a float into an int array and it's chopped with no warning. Mix in one string and every number becomes text. Check `arr.dtype` when numbers look off, and create with `dtype=float` when you need decimals.

# 3. Indexing

1-D indexing works like a list. For 2-D, use **one bracket with a comma**: `m[row, col]`.

In [23]:
marks = np.array([[80, 72, 91],     # Ali
                  [65, 88, 70],     # Sara
                  [95, 60, 84]])    # Hassan
#                  Eng Maths Sci

print(marks[0])          # row 0 - all of Ali's marks
print(marks[1, 2])       # Sara's Science
print(marks[1, 0])      # last student, first subject

[80 72 91]
70
65


# 4. Slicing

Same `start:stop:step` as lists, now per dimension. `:` alone means *all of this axis*.

In [24]:
print(marks[:, 1])       # every row, column 1  -> all Maths marks
print(marks[0:2, :])     # first two students, all subjects
print(marks[:, 1:])      # all students, Maths and Science
print(marks[::2])        # every other student

[72 88 60]
[[80 72 91]
 [65 88 70]]
[[72 91]
 [88 70]
 [60 84]]
[[80 72 91]
 [95 60 84]]


In [ ]:
# WRONG ON PURPOSE - a slice is a VIEW, not a copy
first_row = marks[0]
first_row[0] = 0          # "just changing my copy"...
print(marks)              # ...Ali's English in the original is now 0 too

With lists, a slice is a copy. With NumPy, a slice is a **window onto the same memory** — that's part of why it's fast. If you need an independent copy, say so: `marks[0].copy()`.

In [ ]:
marks[0, 0] = 80                     # put it back
safe = marks[0].copy()
safe[0] = 0
print(marks[0], "<- original untouched")

## Boolean masks — filtering without a loop

A comparison on an array gives an array of `True`/`False`. Use that as an index to keep only the `True` elements. This is how you'll filter data in Pandas tomorrow.

In [ ]:
scores = np.array([45, 82, 67, 91, 38, 74])

print(scores >= 50)                  # the mask
print(scores[scores >= 50])          # only the passes
print((scores >= 50).sum(), "passed")   # True counts as 1

scores[scores < 50] = 50             # grace marks: raise every fail to 50
print(scores)

In [ ]:
# BREAKS ON PURPOSE - 'and' / 'or' don't work on arrays
scores = np.array([45, 82, 67, 91, 38, 74])
scores[scores > 50 and scores < 80]

**ValueError: The truth value of an array with more than one element is ambiguous.** `and` wants one True/False, but you gave it six. For element-wise logic use `&` (and), `|` (or), `~` (not) — **with brackets** around each condition:

In [ ]:
print(scores[(scores > 50) & (scores < 80)])

# 5. Vectorized maths and aggregations

Every arithmetic operator and most maths functions work element-wise. Aggregations like `sum`, `mean`, `max` squash values down — and `axis=` picks the direction:

```
axis=0  ↓  down the rows    -> one result per column  (per subject)
axis=1  →  across columns   -> one result per row     (per student)
```

In [ ]:
marks = np.array([[80, 72, 91],
                  [65, 88, 70],
                  [95, 60, 84]])

print("overall mean     :", marks.mean())
print("per subject mean :", marks.mean(axis=0))
print("per student total:", marks.sum(axis=1))
print("best in each subj:", marks.max(axis=0))
print("who topped each  :", marks.argmax(axis=0))   # row index of the max

In [ ]:
# Element-wise maths functions
x = np.array([1, 4, 9, 16])
print(np.sqrt(x))
print(np.round(np.log(x), 2))
print(np.abs(np.array([-3, 5, -1])))

# 6. Broadcasting — different shapes working together

You already did broadcasting: `prices + 100` added a single number to every element. The rule generalises: NumPy **stretches** the smaller array to match the bigger one when shapes line up from the right.

```
marks        (3, 3)        [[80 72 91]        weights (3,)  [0.3 0.3 0.4]
                            [65 88 70]   *    stretched  →  [0.3 0.3 0.4]
                            [95 60 84]]                     [0.3 0.3 0.4]
```

In [ ]:
weights = np.array([0.3, 0.3, 0.4])          # Eng 30%, Maths 30%, Science 40%

weighted = marks * weights                    # (3,3) * (3,) -> each ROW gets the weights
print(weighted)
print("final grades:", weighted.sum(axis=1))

In [ ]:
# Subtract each subject's average from its column - "centering", a real ML preprocessing step
centered = marks - marks.mean(axis=0)         # (3,3) - (3,)
print(np.round(centered, 1))

In [ ]:
# BREAKS ON PURPOSE - shapes that can't be stretched
bonus = np.array([5, 10])                     # 2 values, but 3 subjects
marks + bonus

**ValueError: operands could not be broadcast together with shapes (3,3) (2,).** Broadcasting only stretches a dimension of size **1** (or a missing one). 2 can't become 3. Once again: `print` the shapes and compare them right-to-left.

# 7. Why AI data is a matrix

Everything a model sees is numbers in an array:

| Data | Array shape |
|------|-------------|
| a table of 1,000 houses, 5 features each | `(1000, 5)` |
| one grayscale 28×28 image | `(28, 28)` |
| a batch of 64 colour 224×224 images | `(64, 224, 224, 3)` |
| a sentence as 10 word-embeddings of size 768 | `(10, 768)` |

And a model's prediction is mostly matrix multiplication — `@` in NumPy. Here's a tiny "linear model" scoring three houses:

In [ ]:
# 3 houses x 3 features: size (100 sq ft), bedrooms, age (years)
houses = np.array([[12, 3, 5],
                   [ 8, 2, 20],
                   [20, 5, 2]])

# The model's learned weights: value (lakh Rs) per unit of each feature
w = np.array([4.0, 10.0, -0.5])
b = 20.0                                     # base price

predictions = houses @ w + b                 # matrix x vector, then broadcast + b
print(predictions)

That one line — `X @ w + b` — is linear regression, which you'll build properly in Month 2. A neural network is the same line stacked in layers. Learn arrays well now, and a lot of ML code will read like plain maths.

In [ ]:
# An "image" is just a 2-D array. Here's an 8x8 one - flip it, invert it, crop it with array ops
img = np.zeros((8, 8), dtype=int)
img[2:6, 2:6] = 1                            # a square in the middle
img[3:5, 3:5] = 0                            # hollow centre

show = lambda a: print("\n".join("".join("#" if p else "." for p in row) for row in a), "\n")
show(img)
show(1 - img)                                 # invert
show(img[1:7, 1:7])                           # crop

# 8. Real-world use case: billing a whole city from smart meters

**The problem.** An electricity company like LESCO has installed smart meters that send a reading **every hour**. For one small area of 5,000 homes, one month of data is:

```
5,000 homes × 30 days × 24 hours = 3,600,000 readings
```

At the end of the month the company has to:

1. **Bill every home**, charging more for units used at **peak hours (6 pm – 10 pm)**
2. Find the **city's peak hour**, so it knows when the grid is most under strain
3. **Flag suspicious meters**: homes whose usage suddenly collapses mid-month, which is a classic sign of meter tampering (electricity theft)

A real company does this for *millions* of homes, every month. Let's see why plain Python struggles and NumPy doesn't.

## The data: one 3-D array

In real life you'd load this from the company's files (`np.load("readings.npy")`). Here we generate realistic fake readings. We also secretly tamper with 12 meters from day 21 onwards, so we can check later whether our detector catches them.

The whole month becomes **one array** of shape `(homes, days, hours)`:

```
usage[h, d, t]  = units (kWh) used by home h, on day d, during hour t
usage[h]        = one home's whole month            shape (30, 24)
usage[:, d]     = every home on day d               shape (5000, 24)
usage[:, :, 19] = every home, every day, at 7 pm    shape (5000, 30)
```

In [ ]:
rng = np.random.default_rng(42)           # seeded, so everyone gets the same "random" data
homes, days, hours = 5000, 30, 24

# A typical day: low at night, a morning bump, a big evening peak (units per hour)
day_shape = np.array([0.4, 0.3, 0.3, 0.3, 0.3, 0.4, 0.7, 0.9, 0.8, 0.6, 0.6, 0.7,
                      0.8, 0.9, 0.9, 0.8, 0.8, 1.0, 1.4, 1.7, 1.8, 1.6, 1.1, 0.6])

house_size = rng.uniform(0.5, 2.0, size=(homes, 1, 1))     # small flat ... big house
noise = rng.uniform(0.7, 1.3, size=(homes, days, hours))   # day-to-day variation
usage = house_size * day_shape * noise                      # broadcasting builds the whole month

tampered = rng.choice(homes, size=12, replace=False)        # 12 secret cheaters
usage[tampered, 20:, :] *= 0.3                              # from day 21, their meter shows 30%

print("shape :", usage.shape)
print("values:", usage.size)
print(f"memory: {usage.nbytes / 1e6:.0f} MB")

## Task 1: the bills, the plain-Python way

Rates (illustrative): **Rs 40** per unit normally, **Rs 55** per unit from 6 pm to 10 pm (hours 18–21). With lists, this means three nested loops and one `if` per reading, 3.6 million times:

In [ ]:
usage_list = usage.tolist()        # the same data as nested Python lists

def bills_with_loops(data):
    bills = []
    for home in data:                          # 5,000 homes
        total = 0
        for day in home:                       #   x 30 days
            for hour, units in enumerate(day): #     x 24 hours
                rate = 55 if 18 <= hour < 22 else 40
                total += units * rate
        bills.append(total)
    return bills

start = time.time()
bills_list = bills_with_loops(usage_list)
loop_time = time.time() - start

print(f"Home 0 bill: Rs {bills_list[0]:,.0f}")
print(f"Time: {loop_time:.2f} s")

## Task 1: the bills, the NumPy way

Now think of it as maths instead of loops:

- Make an array of **24 rates**, one per hour.
- `usage * rates` uses broadcasting: shape `(5000, 30, 24)` times `(24,)` gives every reading multiplied by the rate for its hour.
- `.sum(axis=(1, 2))` adds up days and hours, leaving **one bill per home**.

In [ ]:
rates = np.full(24, 40.0)          # Rs 40 for every hour...
rates[18:22] = 55                  # ...except 6 pm - 10 pm
print(rates)

start = time.time()
bills = (usage * rates).sum(axis=(1, 2))
numpy_time = time.time() - start

print(f"Home 0 bill: Rs {bills[0]:,.0f}")
print(f"Time: {numpy_time:.3f} s  ->  about {loop_time / numpy_time:.0f}x faster")
print("Same answers as the loop version?", np.allclose(bills, bills_list))

**One line instead of nine**, the same answer, and many times faster. Now scale up to LESCO's real size, about 5 million homes, which is 1,000× this data: multiply the loop time above by 1,000. That's minutes of waiting just for bills, run again every time the tariff changes or someone asks a new question. The NumPy version stays at a few seconds, and it's the one line you'd actually trust yourself to change.

Changing the tariff is also trivial: edit the `rates` array. No loop logic to touch.

## Task 2: the city's peak hour

Add up every home and every day, keeping only the hour axis, and ask which hour is biggest:

In [ ]:
city_by_hour = usage.sum(axis=(0, 1))          # shape (24,): total units per hour of the day
peak = city_by_hour.argmax()

print(f"Peak hour: {peak}:00 - {peak + 1}:00")
print(f"Busiest hour uses {city_by_hour.max() / city_by_hour.min():.1f}x the quietest hour")

# A quick text bar chart, one row per hour
for h, units in enumerate(city_by_hour):
    print(f"{h:02d}:00 {'#' * int(units / city_by_hour.max() * 40)}")

## Task 3: catching tampered meters

The idea is to compare each home's **average daily usage** in days 1–20 with days 21–30. Normal homes stay roughly the same. A home that suddenly drops by more than half is suspicious.

Every step is one array operation, done for all 5,000 homes at once:

In [ ]:
daily = usage.sum(axis=2)                      # (5000, 30): units per home per day
before = daily[:, :20].mean(axis=1)            # (5000,): average day, first 20 days
after = daily[:, 20:].mean(axis=1)             # (5000,): average day, last 10 days
drop = 1 - after / before                      # 0.0 = no change, 0.7 = fell by 70%

suspects = np.where(drop > 0.5)[0]             # boolean mask -> home numbers

print("Suspicious meters:", sorted(suspects))
print("Actually tampered:", sorted(tampered))
print("Caught all of them, no false alarms?", set(suspects) == set(tampered))

In [ ]:
# Send the inspection team a short report
for home in suspects[:5]:
    print(f"Home {home:>4}: {before[home]:.1f} -> {after[home]:.1f} units/day "
          f"({drop[home]:.0%} drop)  bill Rs {bills[home]:,.0f}")

## How NumPy solved it

| Need | Plain Python | NumPy |
|------|--------------|-------|
| Hold 3.6M readings | lists inside lists inside lists | one `(5000, 30, 24)` array, 29 MB |
| Different rate per hour | `if` inside a triple loop | a 24-value `rates` array, **broadcast** |
| Total per home / per hour / per day | loops with running totals | `.sum(axis=...)` |
| Find suspicious homes | loop plus `if` plus `append` | a **boolean mask** plus `np.where` |
| Speed at 5 million homes | loop time × 1,000 (minutes) | seconds |

This is the same work a data team does before any AI model is trained: turn raw readings into features (daily usage, drop percentage), then find the unusual ones. A fraud-detection model would take `before`, `after` and `drop` as its input features. Tomorrow's Pandas runs on NumPy arrays underneath, so everything here carries straight over.

# 9. Your turn

**1.** Create a 1-D array of the numbers 1 to 20 with `np.arange`. Print: the even numbers only (boolean mask), their sum, and the array reshaped to `4 × 5`.

**2.** Make a `5 × 4` array of random marks between 30 and 100 (`np.random.randint`) — 5 students, 4 subjects. Print each student's average, each subject's highest mark, and how many individual marks are below 50.

**3.** Temperatures for a week in °C: `[31, 34, 29, 36, 38, 33, 30]`. Convert them all to Fahrenheit (`F = C × 9/5 + 32`) in one line, then print only the days above 95 °F.

**4.** Use the `marks` matrix from Section 5. Give a bonus of `[0, 5, 0]` (5 extra Maths marks) with broadcasting, then cap every mark at 100 using `np.minimum`. Print the result and each student's new weighted final grade using the `weights` from Section 6.

**5. (stretch)** Normalise the `houses` array from Section 7 so every column has mean 0 and standard deviation 1: `(X - X.mean(axis=0)) / X.std(axis=0)`. Check your answer by printing the mean and std of each column of the result.